# Least Privilege for AI Agents, with the Elastic CLI

## Section 1 - Preflight and Provision

In [ ]:
echo "--- tool versions: elastic, claude, jq, yq"
elastic version --json
claude --version
jq --version
yq --version

echo; echo "--- agent skills"
echo "$(ls .agents/skills | wc -l) agent skills installed under .agents/skills/"

echo; echo "--- Elastic Cloud API key from .env (value never printed)"
set -a; source .env; set +a
[[ -n "$EC_API_KEY" ]] && echo "EC_API_KEY loaded" || echo "EC_API_KEY not set - copy .env.sample to .env and fill in your key" >&2

echo; echo "--- CLI config location for this run"
mkdir -p out/
export ELASTIC_CLI_CONFIG_FILE="$PWD/out/elasticrc.yml"
echo "ELASTIC_CLI_CONFIG_FILE=$ELASTIC_CLI_CONFIG_FILE"

echo; echo "--- cloud context: where the Cloud API key was stored"
elastic config context add cloud \
  --cloud-url https://api.elastic-cloud.com \
  --cloud-api-key "$EC_API_KEY" \
  --force --json \
  | jq "{context, action, secret_storage: [.secrets[].storage]}"

echo; echo "--- cloud context: connectivity check"
elastic status --use-context cloud --json

echo; echo "--- create project (progress on stderr, result saved to out/project.json)"
cat assets/config/project.json
elastic cloud serverless projects search create \
  --input-file assets/config/project.json \
  --wait --save-as admin --force --yes \
  --json --output-fields id,name,region_id,endpoints \
  | tee out/project.json

echo; echo "--- contexts now defined, and the admin connectivity check"
elastic config context list --json
elastic status --use-context admin --json

## Section 2 - Load the Catalog

In [ ]:
echo "--- the catalog: line count, then first two documents"
wc -l assets/data/products.ndjson
head -n 2 assets/data/products.ndjson

echo; echo "--- create products (replacing any copy from an earlier run)"
elastic es indices delete --index products --use-context admin --yes --ignore-unavailable --json >/dev/null
elastic es indices create --input-file assets/config/mapping.json --use-context admin --json

echo; echo "--- bulk ingest summary"
elastic es helpers bulk-ingest --index products --data-file assets/data/products.ndjson \
  --use-context admin --json --output-fields total,succeeded,failed

echo; echo "--- document count"
elastic es indices refresh --index products --use-context admin --json >/dev/null
elastic es count --index products --use-context admin --json --output-fields count

## Section 3 - Explore by Hand

In [ ]:
echo "--- status: connectivity and versions"
elastic status --use-context admin --json

echo; echo "--- ES|QL: the catalog at a glance"
elastic es esql query --use-context admin --format tsv \
  --query "FROM products | STATS products = COUNT(*), min_price = MIN(price), max_price = MAX(price), avg_price = ROUND(AVG(price), 2)"

echo; echo "--- search: three most expensive audio products, only the fields that matter"
elastic es search --index products --use-context admin --json --size 3 --sort price:desc \
  --query "{\"term\":{\"category\":\"audio\"}}" \
  --output-fields hits.hits._source.sku,hits.hits._source.name,hits.hits._source.price

echo; echo "--- search: how many products have no price? (--output-template formats the answer)"
elastic es search --index products --use-context admin --size 0 \
  --query "{\"bool\":{\"must_not\":{\"exists\":{\"field\":\"price\"}}}}" \
  --output-template "{{ hits.total.value }} products have no price"

echo; echo "--- ES|QL: documents per category. This should be six rows."
elastic es esql query --use-context admin --format tsv \
  --query "FROM products | STATS docs = COUNT(*) BY category | SORT docs DESC"

## Section 4 - Leash the Agent

In [ ]:
echo "--- layer 1: read-only API key, minted as admin (id and name only)"
AGENT_KEY=$(elastic es security create-api-key --input-file assets/config/agent-key.json --use-context admin --json)
jq "{id, name}" <<<"$AGENT_KEY"

echo; echo "--- agent context: project endpoint plus the read-only key"
elastic config context add agent \
  --es-url "$(jq -r .endpoints.elasticsearch out/project.json)" \
  --es-api-key "$(jq -r .encoded <<<"$AGENT_KEY")" \
  --force --json | jq "{context, action, secret_storage: [.secrets[].storage]}"
unset AGENT_KEY
elastic status --use-context agent --json

echo; echo "--- proof of layer 1: a delete as agent is refused by Elasticsearch"
elastic es indices delete --index products --use-context agent --yes --json 2>&1 \
  | jq -c ".error | {code, status: .status_code, message: .body.error.reason}"


In [ ]:
echo "--- layer 2: command allowlist attached to the agent context, which becomes the default"
yq -i ".contexts.agent.commands = (load(\"assets/config/agent-policy.yml\") | ... comments = \"\")" out/elasticrc.yml
elastic config current-context set agent --json
yq ".contexts.agent.commands.allowed" out/elasticrc.yml

echo; echo "--- proof of layer 2: the same delete, as a bare call, is refused by the CLI before any request is sent"
elastic es indices delete --index products --yes --json 2>&1 \
  | jq -c ".error | {code, message}"
echo "exit code: ${PIPESTATUS[0]}"

echo; echo "--- and a bare read still works"
elastic es count --index products --json --output-fields count

In [ ]:
echo "--- proof of layer 3: the same delete, through Claude Code with --use-context admin, is denied before the binary runs"
claude -p "Run exactly this command and report the result: elastic es indices delete --index products --use-context admin --yes --json" \
  --tools Bash \
  --allowedTools "Bash(elastic *)" \
  --disallowedTools "Bash(elastic * --use-context *)" "Bash(elastic * --config-file *)" \
  --model claude-sonnet-5 --max-turns 3 \
  --output-format stream-json --verbose \
  | jq -r "select(.type==\"user\") | .message.content[]? | select(.type==\"tool_result\") | .content | if type==\"array\" then map(.text // \"\") | join(\"\") else tostring end"

In [ ]:
echo "--- the task"
cat assets/prompts/audit.md

echo; echo "--- agent run: prose as it arrives (full transcript in out/transcript.jsonl)"
claude -p "$(cat assets/prompts/audit.md)" \
  --tools Bash \
  --allowedTools "Bash(elastic *)" \
  --disallowedTools "Bash(elastic * --use-context *)" "Bash(elastic * --config-file *)" \
  --model claude-sonnet-5 --max-turns 30 \
  --output-format stream-json --verbose \
  | tee out/transcript.jsonl \
  | jq -r --unbuffered "select(.type==\"assistant\") | .message.content[]? | select(.type==\"text\") | .text"

echo; echo "--- report saved"
jq -r "select(.type==\"result\") | .result" out/transcript.jsonl > out/report.md
wc -l out/report.md
jq -r "select(.type==\"result\") | \"turns: \\(.num_turns)  duration: \\(.duration_ms / 1000 | floor)s  cost_usd: \\(.total_cost_usd * 100 | round / 100)\"" out/transcript.jsonl

## Section 5 - Apply the Fixes

In [ ]:
echo "--- before acting, two receipts from the transcript: the wall the agent hit (layer 2)"
jq -r "select(.type==\"user\") | .message.content[]? | select(.type==\"tool_result\") | (.content | if type==\"array\" then map(.text // \"\") | join(\"\") else tostring end) | select(test(\"command_blocked\"))" out/transcript.jsonl

echo; echo "--- and no credential string anywhere in it (all must be 0)"
echo "cloud api key: $(grep -c -F "$EC_API_KEY" out/transcript.jsonl)"
echo "project password: $(grep -c -F "$(yq .contexts.admin.elasticsearch.auth.password out/elasticrc.yml)" out/transcript.jsonl)"
echo "agent api key: $(grep -c -F "$(yq .contexts.agent.elasticsearch.auth.api_key out/elasticrc.yml)" out/transcript.jsonl)"

echo; echo "--- the report's sections"
grep "^##" out/report.md

echo; echo "--- before: documents with miscased category, then with untrimmed name"
elastic es esql query --use-context admin --format tsv --query "FROM products | WHERE TO_LOWER(category) != category | STATS docs = COUNT(*)"
elastic es esql query --use-context admin --format tsv --query "FROM products | WHERE name != TRIM(name) | STATS docs = COUNT(*)"

echo; echo "--- fix 1: lowercase category"
elastic es update-by-query --index products --use-context admin --refresh true --json \
  --query "{\"bool\":{\"must_not\":[{\"terms\":{\"category\":[\"audio\",\"cable\",\"case\",\"charger\",\"mount\",\"sensor\"]}}]}}" \
  --script "{\"source\":\"ctx._source.category = ctx._source.category.toLowerCase()\",\"lang\":\"painless\"}" \
  --output-fields total,updated,failures

echo; echo "--- fix 2: trim name"
elastic es update-by-query --index products --use-context admin --refresh true --json \
  --query "{\"regexp\":{\"name.keyword\":\" .*|.* \"}}" \
  --script "{\"source\":\"ctx._source.name = ctx._source.name.trim()\",\"lang\":\"painless\"}" \
  --output-fields total,updated,failures

echo; echo "--- after: the same two counts"
elastic es esql query --use-context admin --format tsv --query "FROM products | WHERE TO_LOWER(category) != category | STATS docs = COUNT(*)"
elastic es esql query --use-context admin --format tsv --query "FROM products | WHERE name != TRIM(name) | STATS docs = COUNT(*)"

echo; echo "--- documents per category, six rows this time"
elastic es esql query --use-context admin --format tsv --query "FROM products | STATS docs = COUNT(*) BY category | SORT docs DESC"

## Section 6 - Teardown

In [ ]:
echo "--- delete the project"
PROJECT_ID=$(jq -r .id out/project.json)
elastic cloud serverless projects search delete --id "$PROJECT_ID" --yes --use-context cloud --json

echo; echo "--- projects remaining"
elastic cloud serverless projects search list --use-context cloud --json | jq -c "[.items[] | {id, name}]"

echo; echo "--- local cleanup"
rm -rf out/ && echo "removed out/"
unset ELASTIC_CLI_CONFIG_FILE